In [1]:
import os
import re
import time
import json
import numpy as np
import pandas as pd
from PIL import Image
import nltk
import tensorflow as tf
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

tf.random.set_seed(42)

# Text constants (same as 03_glove_rnn_text.ipynb)
STOPWORDS = set(stopwords.words('english'))
MAX_LEN   = 50
VOCAB_SIZE = 10000
EMBED_DIM  = 100

# Image constants (same as 03_cnn_mobilenet.ipynb)
IMG_SIZE = (128, 128)
IMG_ROOT = '../data/Images'

# Known best hyperparams from unimodal runs
LSTM_UNITS  = 32   # best from 03_glove_rnn_text.ipynb
DENSE_UNITS = 256  # best from 03_cnn_mobilenet.ipynb

## Build paired DataFrame

Same paired construction as the late-fusion notebook: join LabeledText.csv with image
paths on the numeric file ID (`1.txt` ↔ `1.jpg`). One row per tweet.

In [2]:
folder_to_label = {'Negative': 'negative', 'Neutral': 'neutral', 'positive': 'positive'}
img_records = []
for folder in folder_to_label:
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            fid = int(os.path.splitext(fname)[0])
            img_records.append({'file_id': fid, 'img_path': os.path.join(folder_path, fname)})
img_df = pd.DataFrame(img_records)

text_df = pd.read_csv('../data/LabeledText.csv', encoding='latin-1')
text_df['file_id'] = text_df['File Name'].str.replace('.txt', '', regex=False).astype(int)
text_df['label']   = text_df['LABEL'].str.lower().str.strip()

paired = text_df.merge(img_df, on='file_id')[['file_id', 'Caption', 'label', 'img_path']]
paired = paired.dropna(subset=['Caption', 'img_path']).reset_index(drop=True)

print(f'Paired tweets: {len(paired)}')

Paired tweets: 4869


## Joint train / val / test split

Same split as all unimodal notebooks. The **validation set is used to tune the fusion
weight α** — the test set is touched only once at the end.

In [3]:
trainval_df, test_df = train_test_split(
    paired, test_size=0.2, random_state=42, stratify=paired['label']
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.2, random_state=42, stratify=trainval_df['label']
)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

le = LabelEncoder()
le.fit(paired['label'])
print('Classes:', le.classes_)

y_train    = le.transform(train_df['label'])
y_val      = le.transform(val_df['label'])
y_test     = le.transform(test_df['label'])
y_trainval = le.transform(trainval_df['label'])

Train: 3116  Val: 779  Test: 974
Classes: ['negative' 'neutral' 'positive']


## Text preprocessing

Identical pipeline to `03_glove_rnn_text.ipynb`. Tokenizer fit on joint training set only.
Note we also encode the **val** set this time — needed to tune α.

In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

train_text    = train_df['Caption'].apply(clean_text)
val_text      = val_df['Caption'].apply(clean_text)
test_text     = test_df['Caption'].apply(clean_text)
trainval_text = trainval_df['Caption'].apply(clean_text)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(train_text)

def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_txt    = encode(train_text)
X_val_txt      = encode(val_text)
X_test_txt     = encode(test_text)
X_trainval_txt = encode(trainval_text)

print('Text encoded. Shape:', X_train_txt.shape)

Text encoded. Shape: (3116, 50)


## Load GloVe and build embedding matrix

In [5]:
glove = {}
with open('../data/glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype='float32')
print(f'GloVe vocab: {len(glove):,}')

embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
hits = 0
for word, idx in tokenizer.word_index.items():
    if idx < VOCAB_SIZE and word in glove:
        embedding_matrix[idx] = glove[word]
        hits += 1
print(f'GloVe coverage: {hits} / {min(VOCAB_SIZE, len(tokenizer.word_index))}')

GloVe vocab: 400,000
GloVe coverage: 6660 / 10000


## Load all images into memory

MobileNetV2 `preprocess_input` (scales to [-1, 1]). Loaded once in paired-DataFrame
order so indices align with the text arrays and split masks.

In [6]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype='float32')
    return mobilenet_preprocess(arr)

print(f'Loading {len(paired)} images...')
X_all_img = np.array([load_image(p) for p in paired['img_path']])
print(f'Loaded. Shape: {X_all_img.shape}')

X_train_img    = X_all_img[train_df.index]
X_val_img      = X_all_img[val_df.index]
X_test_img     = X_all_img[test_df.index]
X_trainval_img = X_all_img[trainval_df.index]

print(f'Train img: {X_train_img.shape}  Val: {X_val_img.shape}  Test: {X_test_img.shape}')

Loading 4869 images...
Loaded. Shape: (4869, 128, 128, 3)
Train img: (3116, 128, 128, 3)  Val: (779, 128, 128, 3)  Test: (974, 128, 128, 3)


## Train GloVe + LSTM text model on joint trainval

Architecture identical to `03_glove_rnn_text.ipynb`, `LSTM_UNITS=32`. We capture
softmax probabilities on **both val and test** — val is needed to tune α.

In [7]:
t0 = time.time()

text_model = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], trainable=False),
    LSTM(LSTM_UNITS),
    Dense(3, activation='softmax')
])
text_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
text_model.fit(
    X_trainval_txt, y_trainval,
    epochs=150, batch_size=32,
    callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)],
    verbose=1
)

text_probs_val  = text_model.predict(X_val_txt)
text_probs_test = text_model.predict(X_test_txt)
text_acc = accuracy_score(y_test, np.argmax(text_probs_test, axis=1))
print(f'\nText model accuracy on joint test set: {text_acc:.3f}')

Epoch 1/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.4773 - loss: 1.0140
Epoch 2/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5461 - loss: 0.8764
Epoch 3/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6103 - loss: 0.8173
Epoch 4/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6729 - loss: 0.7662
Epoch 5/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7091 - loss: 0.7161
Epoch 6/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7345 - loss: 0.6663
Epoch 7/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7592 - loss: 0.6213
Epoch 8/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7720 - loss: 0.5945
Epoch 9/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7913 - loss: 0.5647
Epoch 10/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7959 - loss: 0.5476
Epoch 11/150
122/122 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8098 - loss: 0.5240
Epoch 12/150
122/122 ━━━━━━━━━━━━━━━

## Train MobileNetV2 image model on joint trainval

Architecture identical to `03_cnn_mobilenet.ipynb`, `DENSE_UNITS=256`,
`validation_split=0.2` for EarlyStopping. Capture probs on val and test.

In [8]:
base = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base.trainable = False

img_model = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(DENSE_UNITS, activation='relu', kernel_regularizer=l2(1e-4)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])
img_model.compile(optimizer=Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
img_model.fit(
    X_trainval_img, y_trainval,
    validation_split=0.2, epochs=100, batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=1
)

img_probs_val  = img_model.predict(X_val_img)
img_probs_test = img_model.predict(X_test_img)
img_acc = accuracy_score(y_test, np.argmax(img_probs_test, axis=1))
print(f'\nImage model accuracy on joint test set: {img_acc:.3f}')

Epoch 1/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 16s 134ms/step - accuracy: 0.3376 - loss: 1.5313 - val_accuracy: 0.3389 - val_loss: 1.2486
Epoch 2/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 19s 123ms/step - accuracy: 0.3816 - loss: 1.2523 - val_accuracy: 0.3517 - val_loss: 1.1935
Epoch 3/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 13s 135ms/step - accuracy: 0.4278 - loss: 1.1531 - val_accuracy: 0.3530 - val_loss: 1.1764
Epoch 4/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 13s 135ms/step - accuracy: 0.4692 - loss: 1.0968 - val_accuracy: 0.3504 - val_loss: 1.1679
Epoch 5/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 13s 132ms/step - accuracy: 0.5019 - loss: 1.0462 - val_accuracy: 0.3607 - val_loss: 1.1632
Epoch 6/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 14s 138ms/step - accuracy: 0.5157 - loss: 1.0175 - val_accuracy: 0.3671 - val_loss: 1.1613
Epoch 7/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 14s 143ms/step - accuracy: 0.5613 - loss: 0.9601 - val_accuracy: 0.3736 - val_loss: 1.1628
Epoch 8/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 15s 151ms/step - accuracy: 0.5960 - loss: 0.9266 - 

## Tune fusion weight α on the validation set

Fused probability: `α · text_probs + (1 − α) · img_probs`.

α = 1.0 is text-only, α = 0.0 is image-only. We sweep α ∈ [0, 1] in steps of 0.05
and pick the value that maximizes **validation** accuracy — the test set is not consulted.
Because the image model is much weaker, we expect the best α to favor text.

In [9]:
alphas = np.round(np.arange(0.0, 1.0001, 0.05), 2)
sweep = []
for a in alphas:
    fused_val = a * text_probs_val + (1 - a) * img_probs_val
    acc = accuracy_score(y_val, np.argmax(fused_val, axis=1))
    sweep.append({'alpha': a, 'val_acc': acc})
sweep_df = pd.DataFrame(sweep)

best_alpha = sweep_df.loc[sweep_df['val_acc'].idxmax(), 'alpha']
print(f'Best alpha (val): {best_alpha}  (val acc {sweep_df["val_acc"].max():.3f})')
print()
print(sweep_df.to_string(index=False))

Best alpha (val): 0.6  (val acc 0.959)

 alpha  val_acc
  0.00 0.704750
  0.05 0.758665
  0.10 0.838254
  0.15 0.874198
  0.20 0.913992
  0.25 0.943517
  0.30 0.955071
  0.35 0.956354
  0.40 0.956354
  0.45 0.957638
  0.50 0.956354
  0.55 0.957638
  0.60 0.958922
  0.65 0.958922
  0.70 0.958922
  0.75 0.958922
  0.80 0.958922
  0.85 0.958922
  0.90 0.958922
  0.95 0.958922
  1.00 0.958922


## Apply best α to the test set

In [10]:
fused_test = best_alpha * text_probs_test + (1 - best_alpha) * img_probs_test
y_fusion   = np.argmax(fused_test, axis=1)
runtime    = time.time() - t0

fusion_acc = accuracy_score(y_test, y_fusion)
report     = classification_report(y_test, y_fusion, target_names=le.classes_, output_dict=True)

print('=== Joint test set comparison ===')
print(f'  Text only  (GloVe+LSTM):       {text_acc:.3f}')
print(f'  Image only (MobileNetV2):      {img_acc:.3f}')
print(f'  Equal fusion (alpha=0.50):     {accuracy_score(y_test, np.argmax(0.5*text_probs_test + 0.5*img_probs_test, axis=1)):.3f}')
print(f'  Weighted fusion (alpha={best_alpha}):  {fusion_acc:.3f}')
print()
print(classification_report(y_test, y_fusion, target_names=le.classes_))

=== Joint test set comparison ===
  Text only  (GloVe+LSTM):       0.730
  Image only (MobileNetV2):      0.359
  Equal fusion (alpha=0.50):     0.729
  Weighted fusion (alpha=0.6):  0.729

              precision    recall  f1-score   support

    negative       0.69      0.74      0.72       291
     neutral       0.69      0.65      0.67       354
    positive       0.80      0.80      0.80       329

    accuracy                           0.73       974
   macro avg       0.73      0.73      0.73       974
weighted avg       0.73      0.73      0.73       974



## Save metadata

In [11]:
meta = {
    'model': 'multimodal_weighted_fusion',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime,
    'best_alpha': float(best_alpha),
    'text_acc_joint_test': text_acc,
    'img_acc_joint_test': img_acc,
    'fusion_strategy': 'weighted_average_alpha_tuned_on_val',
    'text_model': 'GloVe+LSTM (units=32)',
    'image_model': 'MobileNetV2 (dense=256)'
}

with open('../models/both/json/weighted_fusion_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved. Best alpha={best_alpha}. Total runtime: {runtime:.0f}s ({runtime/60:.1f} min)')

Saved. Best alpha=0.6. Total runtime: 451s (7.5 min)
